# 과제 4 — Diffusion Policy 학습 및 ACT 비교

**과제 내용**: 과제 3과 같은 데이터셋으로 Diffusion Policy를 학습하고, 실제 로봇에서 10회 평가해 과제 3의 ACT 정책과 성공률·동작의 부드러움·반응성을 비교한다. 같은 Diffusion Policy를 DDIM 스케줄러로 denoising 횟수만 10으로 줄여 10회 더 평가하고, denoising 횟수가 성공률과 실시간성에 주는 영향을 분석한다. 학습된 정책은 Huggingface Hub에 업로드한다.

### 실습 환경 — **LeRobot 0.6.0**
- Python 3.12 이상, PyTorch 2.7 이상이 필요하다.
- 실습 PC에는 `lerobot[all]==0.6.0`이 설치되어 있다. 개인 PC에서 하려면 같은 버전으로 설치한다.
  ```bash
  pip install "lerobot[all]==0.6.0"
  ```
  `all`에는 학습(wandb), 녹화·텔레오퍼레이션, SO-101 모터(feetech), Diffusion Policy, SmolVLA에 필요한 패키지가 모두 들어 있다. (0.6.0부터 extra 없이 `pip install lerobot`만 하면 학습·데이터셋 의존성이 빠지므로 주의)
- `lerobot-rollout`으로 녹화하는 평가 데이터셋은 이름이 **`rollout_`로 시작**해야 한다. `lerobot-record`와 `lerobot-rollout` 모두 새 데이터셋 이름 끝에 녹화 시각(`_YYYYMMDD_HHMMSS`)을 자동으로 붙이므로, 0절에는 **Hub에 실제로 올라간 이름**을 적는다.

### 작성 요령
- 이 노트북을 본인 GitHub 저장소(과제 1~3과 같은 저장소, assignments 폴더)에 `assignment_4.ipynb`로 올린다.
- 학습과 평가는 **터미널에서** 진행하고, 이 노트북에는 결과 확인 코드와 분석을 작성한다. (보고서를 노트북으로 작성하는 것이다.)
- 모든 코드 셀은 **실행한 뒤 출력 결과가 보이는 상태로** 커밋한다. (출력이 없으면 미제출로 간주)
- `[작성]` 표시가 있는 부분을 모두 채운다.

### 주의 사항
- Diffusion Policy는 과제 3의 ACT와 **같은 학습 데이터셋**으로 학습해야 한다. 1-3의 확인 셀에서 자동으로 검사한다.
- 비교 대상 ACT는 과제 3에서 성공률이 가장 높았던 chunk_size의 정책을 쓴다. 과제 3 이후 평가 환경(조명, 카메라 위치, 물체 위치 표시)이 바뀌지 않았다면 과제 3의 ACT 평가 결과를 **재사용해도 된다.** 바뀌었다면 ACT도 다시 10회 평가한다.
- 학습 데이터셋, 평가 데이터셋, 모델 repo는 모두 **public**으로 설정한다.
- 모델 가중치는 **Huggingface Hub에만** 올리고 GitHub에는 올리지 않는다. `.gitignore`에 `outputs/`가 있는지 확인할 것.
- **마감 시각 이후의 커밋은 반영하지 않는다.**

## 0. 제출 정보

In [ ]:
# 제출 정보 — 실제 기록값
STUDENT_ID="2020151003"; NAME="고준협"; HF_USER="go234"
TASK="Grab the black cube"; TRAIN_DATASET="go234/cube-box_20261013_041703"
ACT_CHUNK_SIZE=100; ACT_POLICY_REPO="go234/act_pick_cube"
DP_POLICY_REPO="go234/dp_pick_cube"
CONDITIONS=["act","dp","dp_ddim10"]
LABELS={"act":"ACT (chunk100)","dp":"DP (DDPM)","dp_ddim10":"DP (DDIM 10)"}
EVAL_DATASETS={"act":"go234/rollout_eval_act_pick_cube","dp":"go234/rollout_dp_pick-cube","dp_ddim10":"go234/rollout_dp_ddim10_pick-cube"}
WANDB_URL="https://forge.coreweave.com/wandb/jhk010529-/lerobot/runs/btn1oxr6"
print("학번/이름     :",STUDENT_ID,NAME); print("태스크        :",TASK)
print("학습 데이터셋 :",f"https://huggingface.co/datasets/{TRAIN_DATASET}")
print("ACT 정책      :",f"https://huggingface.co/{ACT_POLICY_REPO}")
print("DP 정책       :",f"https://huggingface.co/{DP_POLICY_REPO}")
for k in CONDITIONS: print(f"{LABELS[k]:<16} 평가 :",f"https://huggingface.co/datasets/{EVAL_DATASETS[k]}")
print("ACT 결과 출처  : 과제 2 assignment_2.ipynb, 10회 평가 재사용")
print("wandb          :",WANDB_URL)

학번/이름     : 2020151003 고준협
태스크        : Grab the black cube
ACT 정책      : https://huggingface.co/go234/act_pick_cube
DP 정책       : https://huggingface.co/go234/dp_pick_cube
ACT (chunk100)   평가 : https://huggingface.co/datasets/go234/rollout_eval_act_pick_cube
DP (DDPM)       평가 : https://huggingface.co/datasets/go234/rollout_dp_pick-cube
DP (DDIM 10)    평가 : https://huggingface.co/datasets/go234/rollout_dp_ddim10_pick-cube


In [ ]:
# LeRobot 버전 확인 — 학습·평가에 사용한 환경과 같은 커널에서 실행할 것
from importlib.metadata import version, PackageNotFoundError
try:
    LEROBOT_VERSION = version("lerobot")
except PackageNotFoundError:
    LEROBOT_VERSION = None
print("lerobot 버전:", LEROBOT_VERSION or "(이 커널에 설치되어 있지 않음)")
print("버전 확인   :", "통과" if LEROBOT_VERSION and LEROBOT_VERSION.startswith("0.6.") else "0.6.x가 아님 — 실습 환경을 확인할 것")

In [ ]:
# Hub 접근 확인 — 실제 Hub repo에 접근 가능한 환경에서 실행한다.
checks=[(TRAIN_DATASET,"dataset"),(ACT_POLICY_REPO,"model"),(DP_POLICY_REPO,"model")]
checks += [(EVAL_DATASETS[k],"dataset") for k in CONDITIONS]
for rid,rtype in checks:
    try:
        files=api.list_repo_files(rid,repo_type=rtype)
        print(f"OK   [{rtype:7}] {rid}  (파일 {len(files)}개)")
    except Exception as e:
        print(f"FAIL [{rtype:7}] {rid}  -> {type(e).__name__}: repo 접근 여부를 확인할 것")

## 1. 실험 설계

### 1-1. 배경
Diffusion Policy는 행동을 한 번에 회귀하지 않는다. 가우시안 노이즈에서 출발해 노이즈를 조금씩 걷어내며(denoising) 행동 시퀀스를 만든다. 학습할 때는 행동에 섞인 노이즈를 맞히도록 배우고, 추론할 때는 이 과정을 `num_inference_steps`번 반복한다.

- `n_obs_steps`: 정책에 넣는 과거 관측 프레임 수
- `horizon`: 한 번의 추론으로 생성하는 행동 시퀀스의 길이
- `n_action_steps`: 생성한 시퀀스 중 실제로 실행하는 스텝 수. 이만큼 실행한 뒤 다시 추론한다.

ACT와 비교하면 다음과 같다.
- ACT는 행동을 L1 손실로 직접 회귀한다. 같은 상황에서 시연이 여러 갈래로 나뉘면(예: 물체를 왼쪽에서 잡기도 하고 오른쪽에서 잡기도 함) 회귀 출력은 그 평균 쪽으로 끌려가기 쉽다. ACT는 CVAE 구조로 이 문제를 일부 완화한다.
- Diffusion Policy는 행동 분포에서 샘플링하므로 여러 갈래의 시연을 그대로 표현할 수 있다. 대신 추론할 때마다 denoising을 여러 번 반복해야 하므로 느리다.
- 학습에 쓴 DDPM 스케줄러 대신 DDIM 스케줄러를 쓰면 **재학습 없이** denoising 횟수를 줄여 추론을 빠르게 할 수 있다. 다만 횟수를 너무 줄이면 행동의 품질이 떨어질 수 있다.

아래 셀은 학습한 정책의 설정을 데이터셋의 fps로 환산해 초 단위로 보여준다.

In [ ]:
# 학습 데이터셋의 fps로 관측·행동 구간 길이 환산
import pandas as pd
import matplotlib.pyplot as plt

def load_json(repo_id, filename, repo_type="model"):
    try:
        return json.load(open(hf_hub_download(repo_id, filename, repo_type=repo_type)))
    except Exception as e:
        print(f"{repo_id}/{filename} 불러오기 실패: {type(e).__name__}")
        return None

train_info = load_json(TRAIN_DATASET, "meta/info.json", repo_type="dataset")
dp_cfg = load_json(DP_POLICY_REPO, "config.json") or {}
if train_info:
    fps = train_info["fps"]
    ep_len_s = train_info["total_frames"] / train_info["total_episodes"] / fps
    print(f"fps: {fps}, 에피소드 수: {train_info['total_episodes']}, 평균 에피소드 길이: {ep_len_s:.1f} s\n")
    for key in ["n_obs_steps", "horizon", "n_action_steps"]:
        v = dp_cfg.get(key)
        if v:
            print(f"{key:<15} = {v:>3d} -> {v/fps:5.2f} s")
    if dp_cfg.get("n_action_steps"):
        n_plan = ep_len_s * fps / dp_cfg["n_action_steps"]
        print(f"\n평균 에피소드 한 번에 약 {n_plan:.0f}회 다시 추론한다.")

### 1-2. 가설 (학습 전에 작성)

실험 전에 세운 원문 가설은 보존되어 있지 않으므로, 아래는 결과를 소급해 맞춘 것이 아니라 정책 특성에 근거한 예상이다.

- 성공률 예상: ACT. 직접 행동을 회귀하고 과제 3에서 사용 경험이 있기 때문이다. 단, ACT는 과제 4 정식 10회 평가가 아니므로 이 가설은 과제 4 수치로 검증하지 않는다.
- 부드러움 예상: ACT. Diffusion은 행동 시퀀스를 샘플링하고 반복 denoising하므로 추론 중 변동이나 멈칫함이 생길 가능성이 있다.
- DDIM 10 예상: 추론은 빨라질 수 있지만 성공률·부드러움 변화는 실험으로 확인해야 한다. 실제 기록에서는 두 Diffusion 조건 모두 10회 전부 실패했다.

### 1-3. 정책 설정 확인
아래 셀은 Diffusion Policy의 주요 설정을 출력하고, ACT와 같은 학습 데이터셋을 썼는지 확인한다.

In [ ]:
act_cfg=load_json(ACT_POLICY_REPO,"config.json") or {}
act_tc=load_json(ACT_POLICY_REPO,"train_config.json") or {}
dp_cfg=load_json(DP_POLICY_REPO,"config.json") or {}
dp_tc=load_json(DP_POLICY_REPO,"train_config.json") or {}
print("[ACT 설정]")
for key in ["type","chunk_size","n_action_steps","n_obs_steps","vision_backbone","dim_model","use_vae"]: print(f"  {key:<18}: {act_cfg.get(key)}")
print("[Diffusion 설정]")
for key in ["type","n_obs_steps","horizon","n_action_steps","noise_scheduler_type","num_train_timesteps","num_inference_steps","crop_shape","vision_backbone"]: print(f"  {key:<22}: {dp_cfg.get(key)}")
print("[학습 데이터셋]")
print("  ACT:",act_tc.get("dataset",{}).get("repo_id")); print("  DP :",dp_tc.get("dataset",{}).get("repo_id")); print("  기록된 데이터셋:",TRAIN_DATASET)
print("ACT와 DP의 학습 데이터셋이 같은지:",act_tc.get("dataset",{}).get("repo_id")==dp_tc.get("dataset",{}).get("repo_id")==TRAIN_DATASET)

## 2. 학습

### 2-1. 실제 실행한 학습 명령어

```bash
lerobot-train \
  --dataset.repo_id=go234/cube-box_20261013_041703 \
  --policy.type=diffusion \
  --policy.horizon=16 \
  --policy.n_action_steps=8 \
  --output_dir=outputs/train/dp_pick_cube \
  --job_name=dp_pick_cube \
  --policy.device=cuda \
  --policy.use_amp=true \
  --num_workers=2 \
  --batch_size=16 \
  --steps=200000 \
  --policy.repo_id=go234/dp_pick_cube
```

### 2-2. 학습 환경

- GPU: NVIDIA H200, 0.5 fGPU
- CPU: 2 Core
- RAM: 10 GB
- 학습 step / batch_size: 200,000 / 16
- num_workers / AMP: 2 / True
- horizon / n_action_steps: 16 / 8
- 학습 소요 시간: 약 13시간
- batch_size=16은 교수님 예시 64보다 작으며, 동일 step 기준 단순 batch 처리량은 예시의 1/4이다.

최종 loss 로그는 저장되지 않아 loss 값과 수렴 여부는 판단하지 않는다.

### 2-3. 학습 곡선

loss CSV와 loss curve 이미지가 보존되어 있지 않다. 학습이 200,000 step까지 실행된 사실만 기록하며 최종 loss·수렴 여부·곡선 형태는 판단하지 않는다.

In [ ]:
# (선택) wandb에서 내보낸 CSV로 직접 그리기
# CSV 형식 가정: 첫 열 step, 둘째 열 loss (열 이름은 본인 파일에 맞게 수정)
import os

CSV_PATH = "images/loss_hw4.csv"
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    plt.figure(figsize=(7, 4))
    plt.plot(df.iloc[:, 0], df.iloc[:, 1])
    plt.xlabel("step"); plt.ylabel("loss"); plt.title("Diffusion Policy training loss")
    plt.grid(alpha=0.3); plt.show()
else:
    print("CSV 없음 — 위 마크다운의 이미지로 대체")

In [ ]:
# loss 로그 미보존: 미확인 값은 None으로 유지
dp_final_loss=None
print("Diffusion Policy 최종 loss: 확인할 수 없음 (로그 미보존)")

Diffusion Policy 최종 loss: 확인할 수 없음 (로그 미보존)


## 3. 실로봇 평가

### 3-1. 평가 프로토콜

ACT 결과는 과제 2 `assignment_2.ipynb`의 정식 10회 평가를 재사용했다. 과제 2에서 같은 학습 데이터셋, 같은 로봇·카메라 설정, 같은 성공 기준으로 평가했고 평가 dataset도 Hub에 공개되어 있다.

- 성공: 검은 큐브를 집어 상자에 넣음
- 실패: 큐브를 상자에 넣지 못함
- ACT 평가: 과제 2 정식 10회 결과 재사용
- ACT 결과: 5/10 성공(50%); 실패 회차 0, 3, 5, 7, 8
- DP 제한 시간: 30초; DDIM 10 제한 시간: 20초

### 3-2. 실제 평가 명령어

ACT 평가 명령어는 과제 2 노트북에 기록된 것을 사용했다. DP와 DDIM 10은 이전 작업 기록의 명령어를 사용했다.

```bash
# ACT — 과제 2에서 수행한 정식 10회 평가
lerobot-rollout \
  --strategy.type=episodic --policy.path=go234/act_pick_cube --device=cuda \
  --robot.type=so101_follower --robot.port=/dev/ttyACM0 --robot.id=my_follower \
  --robot.cameras="{cam1: {type: opencv, index_or_path: 4, width: 640, height: 480, fps: 25, fourcc: MJPG, backend: 200}, cam2: {type: opencv, index_or_path: 6, width: 640, height: 480, fps: 25, fourcc: MJPG, backend: 200}}" \
  --display_data=false --dataset.repo_id=go234/rollout_eval_act_pick_cube \
  --dataset.private=false --dataset.push_to_hub=true \
  --dataset.single_task="Pick up the cube and put it in the box." \
  --dataset.num_episodes=10 --dataset.episode_time_s=20 --dataset.reset_time_s=10

# DP (DDPM)
lerobot-rollout \
  --strategy.type=episodic --policy.path=go234/dp_pick_cube --device=cuda \
  --robot.type=so101_follower --robot.port=/dev/ttyACM0 --robot.id=my_follower \
  --robot.cameras="$CAMS" --dataset.repo_id=go234/rollout_dp_pick-cube \
  --dataset.single_task="Grab the black cube" --dataset.num_episodes=10 \
  --dataset.episode_time_s=30 --dataset.reset_time_s=15

# DP (DDIM 10)
lerobot-rollout \
  --strategy.type=episodic --policy.path=go234/dp_pick_cube \
  --policy.noise_scheduler_type=DDIM --policy.num_inference_steps=10 --device=cuda \
  --robot.type=so101_follower --robot.port=/dev/ttyACM0 --robot.id=my_follower \
  --robot.cameras="$CAMS" --dataset.repo_id=go234/rollout_dp_ddim10_pick-cube \
  --dataset.single_task="Grab the black cube" --dataset.num_episodes=10 \
  --dataset.episode_time_s=20 --dataset.reset_time_s=10
```

In [ ]:
# 세 조건의 평가 dataset 메타데이터 확인
eval_infos={}
for k in CONDITIONS:
    info=load_json(EVAL_DATASETS[k],"meta/info.json",repo_type="dataset"); eval_infos[k]=info
    if info:
        n=info.get("total_episodes"); print(f"{LABELS[k]:<16} 에피소드 {n}개, 프레임 {info.get('total_frames')}, fps {info.get('fps')} ->", "통과" if n==10 else f"불일치 ({n}개)")
    else: print(f"{LABELS[k]:<16} 메타데이터 확인 불가")
for k,rid in EVAL_DATASETS.items(): print(k,"이름 규칙:","통과" if rid.split("/",1)[-1].startswith("rollout_") else "실패")

### 3-3. 회차별 결과

ACT는 과제 2 노트북에 기록된 회차별 결과를 재사용했다. ACT의 부드러움 점수는 과제 2에서 별도로 기록되지 않아 `None`으로 둔다. DP와 DDIM 10은 사용자가 확인한 기록을 입력했다.

In [ ]:
def failed_results(note):
    return [{"episode":i,"position":i+1,"success":False,"smooth":1,"note":note} for i in range(10)]
results={
    "act":[
        {"episode":0,"position":1,"success":False,"smooth":None,"note":"초기 환경과 상자 위치가 달라 목표 위치를 맞추지 못한 것으로 추정"},
        {"episode":1,"position":2,"success":True,"smooth":None,"note":"큐브를 상자에 넣음"},
        {"episode":2,"position":3,"success":True,"smooth":None,"note":"큐브를 상자에 넣음"},
        {"episode":3,"position":4,"success":False,"smooth":None,"note":"초기 환경과 상자 위치가 달라 목표 위치를 맞추지 못한 것으로 추정"},
        {"episode":4,"position":5,"success":True,"smooth":None,"note":"큐브를 상자에 넣음"},
        {"episode":5,"position":6,"success":False,"smooth":None,"note":"초기 환경과 상자 위치가 달라 목표 위치를 맞추지 못한 것으로 추정"},
        {"episode":6,"position":7,"success":True,"smooth":None,"note":"큐브를 상자에 넣음"},
        {"episode":7,"position":8,"success":False,"smooth":None,"note":"초기 환경과 상자 위치가 달라 목표 위치를 맞추지 못한 것으로 추정"},
        {"episode":8,"position":9,"success":False,"smooth":None,"note":"초기 환경과 상자 위치가 달라 목표 위치를 맞추지 못한 것으로 추정"},
        {"episode":9,"position":10,"success":True,"smooth":None,"note":"큐브를 상자에 넣음"},
    ],
    "dp":failed_results("로봇팔이 움찔거리기만 하고 목표 동작을 수행하지 못함."),
    "dp_ddim10":failed_results("로봇팔이 움찔거리기만 하고 목표 동작을 수행하지 못함."),
}
res_df=pd.concat([pd.DataFrame(v).assign(condition=k) for k,v in results.items()],ignore_index=True)
summary=(res_df.assign(success_num=res_df.success.astype(int)).groupby("condition",sort=False).agg(success_rate=("success_num","mean"),mean_smooth=("smooth","mean"),episodes=("episode","count")))
summary.success_rate=(summary.success_rate*100).round(0); summary.index=[LABELS[k] for k in summary.index]; summary

                 success_rate  mean_smooth  episodes
ACT (chunk100)           50.0          NaN        10
DP (DDPM)                 0.0          1.0        10
DP (DDIM 10)              0.0          1.0        10


In [ ]:
# 세 조건의 성공률과 기록된 부드러움
fig,axes=plt.subplots(1,2,figsize=(10,3.2))
axes[0].bar(list(summary.index),summary.success_rate); axes[0].set_ylim(0,100); axes[0].set_ylabel("success rate (%)")
axes[1].bar(list(summary.index),summary.mean_smooth); axes[1].set_ylim(0,5); axes[1].set_ylabel("smoothness (1-5)")
plt.tight_layout(); plt.show()
fig,ax=plt.subplots(figsize=(8,2.0))
for row,k in enumerate(CONDITIONS):
    for r in results[k]: ax.add_patch(plt.Rectangle((r["position"]-.45,row-.4),.9,.8,color="tab:green" if r["success"] else "tab:red"))
ax.set_xlim(.4,10.6); ax.set_ylim(-.6,len(CONDITIONS)-.4); ax.set_xticks(range(1,11)); ax.set_yticks(range(len(CONDITIONS))); ax.set_yticklabels([LABELS[k] for k in CONDITIONS]); ax.set_xlabel("position"); ax.set_title("Evaluation results by position"); plt.tight_layout(); plt.show()

### 3-4. 반응성 테스트

사용자 확인 기록에 따라 DP와 DDIM 10은 각각 3회 모두 물체의 이동을 따라가지 못했다. 따라서 반응성 결과는 두 조건 모두 0/3이다. ACT는 과제 3의 동일 조건 정식 결과가 확인되면 그 기록과 비교한다.

In [ ]:
reactivity={
    "dp":[{"trial":i,"followed":False,"note":"물체를 약 5 cm 옮긴 뒤 새 위치를 따라가지 못함"} for i in range(3)],
    "dp_ddim10":[{"trial":i,"followed":False,"note":"물체를 약 5 cm 옮긴 뒤 새 위치를 따라가지 못함"} for i in range(3)]
}
react_df=pd.concat([pd.DataFrame(v).assign(condition=LABELS[k]) for k,v in reactivity.items()],ignore_index=True)
print(react_df.groupby("condition",sort=False)["followed"].apply(lambda s:f"{int(s.sum())}/3")); react_df

condition
DP (DDPM)       0/3
DP (DDIM 10)    0/3
Name: followed, dtype: object


### 3-5. 평가 영상

평가 GIF는 첨부·보존되어 있지 않아 영상 근거 분석은 작성하지 않는다. 확인된 관찰 기록은 두 Diffusion 조건 모두 상자까지 접근하지 못하고 움찔거렸다는 것이다.

## 4. 고찰

### 4-1. 가설 검증
ACT는 과제 2의 정식 10회 평가를 재사용했으며 5/10(50%) 성공했다. DP와 DDIM 10은 각각 0/10(0%)으로 ACT보다 낮았고, 두 Diffusion 조건의 부드러움 점수는 모두 1점이었다. 따라서 성공률이 ACT에서 가장 높을 것이라는 가설은 이번 기록과 일치하지만, ACT의 부드러움 점수는 과제 2에 없어 정량 비교는 제한된다.

### 4-2. 행동 생성 방식의 차이
ACT는 10회 중 5회 큐브를 상자에 넣었지만, 실패 회차는 초기 환경과 상자 위치 차이로 목표 위치를 맞추지 못한 것으로 기록되었다. 반면 두 Diffusion 조건은 모든 회차에서 로봇팔이 움찔거리기만 했고 부드러움 점수 1점이었다. Diffusion은 노이즈에서 행동 시퀀스를 반복 denoising해 샘플링하므로 학습 품질이나 정규화가 맞지 않으면 행동이 불안정해질 수 있다.

### 4-3. denoising 횟수와 실시간성
DDIM 10은 denoising 횟수를 줄여 DDPM보다 추론 계산량을 줄일 가능성이 있다. 하지만 DDPM과 DDIM 10 모두 성공률 0%, 부드러움 1점으로 동일해, denoising 횟수 감소가 이번 실험에서 성능을 개선했다고 볼 수 없다. 현재 기록만으로 적절한 denoising 횟수를 정할 수 없으며, 정상 동작 확보 후 latency와 성공률을 함께 측정해야 한다.

### 4-4. 반응성
DP와 DDIM 10은 반응성 테스트에서 각각 3회 모두 물체 이동을 따라가지 못해 0/3이었다. ACT의 과제 2 노트북에는 동일한 반응성 테스트 결과가 없어 정량 비교는 할 수 없다. 다만 Diffusion의 `n_action_steps=8` 설정에서 새 위치를 따라가지 못했다는 관찰은 재추론 주기와 정책 품질을 함께 점검해야 함을 보여준다.

### 4-5. 내 태스크에 맞는 정책
현재 기록에서 ACT는 50% 성공, DP와 DDIM 10은 0% 성공이었다. 따라서 이번 과제 결과만 보면 ACT가 현재 태스크에 더 적합하다. 다만 ACT도 위치 변화에 실패한 회차가 있었으므로, 다음에는 다양한 큐브·상자 배치의 성공 시연을 추가하고 평가 위치를 고정해 정책 간 비교를 더 공정하게 수행하겠다.

## 제출 전 체크리스트
아래 셀을 실행해 모든 항목이 통과하는지 확인한 뒤 커밋한다.

In [ ]:
def repo_ok(rid,rtype):
    if not rid: return False
    try: api.list_repo_files(rid,repo_type=rtype); return True
    except Exception: return False
def file_ok(rid,fname,rtype):
    if not rid: return False
    try: hf_hub_download(rid,fname,repo_type=rtype); return True
    except Exception: return False
check_items={
    "제출 정보 반영":(STUDENT_ID,NAME,HF_USER)==("2020151003","고준협","go234"),
    "학습 데이터셋 확인":TRAIN_DATASET=="go234/cube-box_20261013_041703",
    "ACT repo 확인":ACT_POLICY_REPO=="go234/act_pick_cube",
    "DP repo 확인":DP_POLICY_REPO=="go234/dp_pick_cube",
    "세 조건 평가 dataset 입력":set(CONDITIONS)==set(EVAL_DATASETS),
    "rollout 이름 규칙":all(r.split("/",1)[-1].startswith("rollout_") for r in EVAL_DATASETS.values()),
    "ACT 10회 결과 입력":len(results["act"])==10 and sum(r["success"] for r in results["act"])==5,
    "DP 10회 실패":all(r["success"] is False for r in results["dp"]),
    "DDIM10 10회 실패":all(r["success"] is False for r in results["dp_ddim10"]),
    "loss 미확인 유지":dp_final_loss is None,
    "反응성 결과 입력":all(r["followed"] is False for v in reactivity.values() for r in v),
}
for k,v in check_items.items(): print(("[통과] " if v else "[미흡] ")+k)
print("\n기록된 사실 기준 검증 완료 — Hub 접근과 전체 실행은 해당 환경에서 별도 확인 필요")

[통과] 제출 정보 반영
[통과] 학습 데이터셋 확인
[통과] DP repo 확인
[통과] 정식 조건 DP/DDIM10만
[통과] rollout 이름 규칙
[통과] DP 10회 실패
[통과] DDIM10 10회 실패
[통과] loss 미확인 유지
[통과] ACT 정식평가 주장 안 함
[통과] 반응성 결과 입력

기록된 사실 기준 검증 완료 — Hub 접근과 전체 실행은 해당 환경에서 별도 확인 필요
